In [1]:
#Cell 1: 基础导入与 A100 优化设置S

In [2]:
import re
import gc
import json
import math
import os

import matplotlib.pyplot as plt
import numpy as np
import torch
import yaml
from datasets import load_dataset, load_from_disk
from tqdm.notebook import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer

json_path = "./results/pruning_config.json"
dataset_path = "/liucw/Pruning/dataset_wikitext"
lcb_path = "./results/lcb_scores_v4.pt"
# Llama-2-7b Llama-3.1-8B Llama-3-8B
model_path = "./models/Llama-2-7b" 


device = "cuda" if torch.cuda.is_available() else "cpu"
dtype = torch.bfloat16  

print(f"当前设备: {torch.cuda.get_device_name(0)} | 显存总量: {torch.cuda.get_device_properties(0).total_memory/1024**3:.2f} GB")

当前设备: NVIDIA A100 80GB PCIe | 显存总量: 79.15 GB


In [3]:
#Cell 2: 加载原始模型

In [4]:
#  加载 Tokenizer 并处理垫片词
print("正在加载 Tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(model_path)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# 加载大模型
print("正在加载 Model ...")
model = AutoModelForCausalLM.from_pretrained(
    model_path,
    torch_dtype=torch.bfloat16,
    device_map="auto"
)

# 强制开启所有参数的梯度计算 (拦截 DCT 频域特征的刚需！)
for param in model.parameters():
    param.requires_grad = True

# 5. 加载校准集
print("正在加载校准数据集...")
dataset = load_from_disk(dataset_path)
print(f"模型加载成功，占用显存: {torch.cuda.memory_allocated()/1024**3:.2f} GB")

正在加载 Tokenizer...


`torch_dtype` is deprecated! Use `dtype` instead!


正在加载 Model ...


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

正在加载校准数据集...
模型加载成功，占用显存: 12.55 GB


In [5]:
#任务贡献分析 Cell 3: 定义频域分析引擎

In [6]:
class TaskContributionAnalyzer:
    def __init__(self, num_bands=5, num_buckets=20):
        self.B = num_bands
        self.Q = num_buckets

    def bf16_dct_ii(self, x):
        N = x.shape[-1]
        x_pad = torch.cat([x[..., ::2], x[..., 1::2].flip(dims=[-1])], dim=-1)
        M = 2 ** (int(math.ceil(math.log2(N)))) 
        if M > N: x_pad = torch.nn.functional.pad(x_pad, (0, M - N))
        
        # A100 上 FFT 在 fp32 下最稳
        X_fft = torch.fft.rfft(x_pad.float(), dim=-1)
        k = torch.arange(X_fft.shape[-1], device=x.device, dtype=torch.float32)
        phi = torch.exp(-1j * math.pi * k / (2 * N))
        return (2 * (X_fft * phi).real)[..., :N].to(x.dtype)

    def get_spectrum(self, grads):
        # 这里的 grads 是从 Hook 中获取的
        g = torch.norm(grads, p=2, dim=-1)
        g_tilde = (g - g.mean()) / (g.std() + 1e-8)
        energy = torch.abs(self.bf16_dct_ii(g_tilde))**2
        # 自适应池化到 Q 个桶
        z = torch.nn.functional.adaptive_avg_pool1d(energy.view(1, 1, -1), self.Q)
        return z.view(-1)

analyzer = TaskContributionAnalyzer()

In [7]:
#Cell 4: 带有层级保护的配置生成函数 稳健剪枝决策 
# - 保护层：100% 保留
#- 非保护层：根据 LCB 分数在平民层中的相对位置，动态分配 0.2 到 0.5 的保底率

In [8]:
def generate_config(lcb_scores_dict, target_sparsity=0.5, min_head_survival_rate=0.1, num_layers=32, num_heads=32, **kwargs):
    # 自动处理输入是列表的情况
    if isinstance(lcb_scores_dict, list):
        # 如果是列表，我们假设它已经按分数排好序了，或者我们将其视为权重相等的集合
        all_keys = lcb_scores_dict
        # 创建一个伪字典用于后续排序逻辑
        scores_map = {k: (len(all_keys) - i) for i, k in enumerate(all_keys)}
    else:
        all_keys = list(lcb_scores_dict.keys())
        scores_map = lcb_scores_dict

    total_units = len(all_keys)
    target_retain_count = int(total_units * (1 - target_sparsity)) 
    selected_ids = []
    
    # 1. 强制保留所有 MLP 层 (这是防止 PPL 飙升的核心)
    mlp_keys = [k for k in all_keys if 'mlp' in k and 'channel' not in k]
    selected_ids.extend(mlp_keys)
    
    # 2. 提取 Attention Head 键
    head_keys = [k for k in all_keys if 'head' in k]
    
    # 3. 覆盖约束：每层保底
    min_keep_per_layer = max(1, int(num_heads * min_head_survival_rate))
    
    for l_idx in range(num_layers):
        layer_heads = [k for k in head_keys if f"layer_{l_idx}_" in k]
        # 使用 scores_map 进行排序
        layer_heads_sorted = sorted(layer_heads, key=lambda k: scores_map.get(k, 0), reverse=True)
        
        safeguard = layer_heads_sorted[:min_keep_per_layer]
        selected_ids.extend(safeguard)
        for h in safeguard:
            if h in head_keys: head_keys.remove(h)
    
    # 4. 剩余名额全局竞争
    remaining_quota = target_retain_count - len(selected_ids)
    if remaining_quota > 0:
        remaining_heads_sorted = sorted(head_keys, key=lambda k: scores_map.get(k, 0), reverse=True)
        selected_ids.extend(remaining_heads_sorted[:remaining_quota])
        
    return selected_ids

In [9]:
#Cell 5: 物理权重清零  物理执行与交互式评估

In [10]:
def apply_pruning(model, selected_ids):
    """
    智能原位剪枝函数 (终极版：支持 Attention Head + MLP 通道级/结构级精准切除)
    能够自动解析多种格式的 selected_ids。
    
    支持的格式示例:
        - 头级剪枝: 'layer_31_head_18'
        - 伪装头剪枝: 'layer_17_head_32' (将自动识别为保留整层 MLP)
        - 结构级MLP: 'mlp_12' 或 'layer_12_mlp'
        - 通道级MLP: 'layer_12_mlp_channel_502'
    """
    
    # ==========================================
    # 🧠 第一步：智能解析，将一维的名字列表转换为结构化的字典
    # ==========================================
    retention_dict = {}
    for i in range(model.config.num_hidden_layers):
        # 初始状态：默认什么都不留，只有在 selected_ids 里的才配活下来
        retention_dict[i] = {
            'heads': [],            # 保留的注意力头索引
            'mlp_full_keep': False, # 是否整层满血保留 MLP
            'mlp_channels': []      # 精细保留的 MLP 通道索引
        }
        
    for uid in selected_ids:
        uid_lower = str(uid).lower()
        parts = uid_lower.split('_')
        
        try:
            # 1. 处理 Attention Head 以及伪装成 Head 的 MLP
            if 'head' in uid_lower:
                l_idx = int(parts[parts.index('layer') + 1])
                h_idx = int(parts[parts.index('head') + 1])
                
                if h_idx == 32:
                    retention_dict[l_idx]['mlp_full_keep'] = True
                else:
                    retention_dict[l_idx]['heads'].append(h_idx)
                    
            # 2. 处理精细的 MLP 通道 (Channel-level)
            elif 'channel' in uid_lower:
                l_idx = int(parts[parts.index('layer') + 1])
                c_idx = int(parts[parts.index('channel') + 1])
                retention_dict[l_idx]['mlp_channels'].append(c_idx)
                
            # 3. 处理整层 MLP (结构级向后兼容)
            elif 'mlp' in uid_lower and 'channel' not in uid_lower:
                if 'layer' in parts:
                    l_idx = int(parts[parts.index('layer') + 1])
                else:
                    l_idx = int([p for p in parts if p.isdigit()][0])
                retention_dict[l_idx]['mlp_full_keep'] = True
                
        except (ValueError, IndexError):
            continue # 忽略解析失败的异常ID

    # ==========================================
    # ✂️ 第二步：准备大模型的架构参数
    # ==========================================
    config = model.config
    num_heads = config.num_attention_heads
    num_kv_heads = config.num_key_value_heads
    head_dim = config.hidden_size // num_heads
    num_groups = num_heads // num_kv_heads  # GQA 的组数
    intermediate_size = config.intermediate_size # MLP 的总通道数 (如 11008)
    
    total_heads_pruned = 0
    total_mlp_channels_pruned = 0
    
    print("\n" + "="*70)
    print("准备就绪，开始执行智能原位剪枝 (Attention + Channel-level MLP)...")
    print("="*70)

    # ==========================================
    # 🩸 第三步：逐层执行物理手术
    # ==========================================
    for l_idx, layer in enumerate(model.model.layers):
        # --- 提取当前层的保留配置 ---
        heads_to_keep = sorted(set(retention_dict[l_idx]['heads']))
        heads_to_prune = [h for h in range(num_heads) if h not in heads_to_keep]
        
        # ------------------------------------------
        # 手术 A：切除 Attention Heads
        # ------------------------------------------
        if len(heads_to_prune) == num_heads:
            # 如果这层一个头都不留，直接把权重全清零
            layer.self_attn.q_proj.weight.data.zero_()
            layer.self_attn.k_proj.weight.data.zero_()
            layer.self_attn.v_proj.weight.data.zero_()
            layer.self_attn.o_proj.weight.data.zero_()
            total_heads_pruned += num_heads
            
        elif len(heads_to_prune) > 0:
            # 精细切片清零
            # Q_proj 处理
            q_w = layer.self_attn.q_proj.weight.data.view(num_heads, head_dim, -1)
            for h in heads_to_prune:
                q_w[h].zero_()
            
            # K_proj 和 V_proj 处理 (考虑 GQA 分组)
            k_w = layer.self_attn.k_proj.weight.data.view(num_kv_heads, head_dim, -1)
            v_w = layer.self_attn.v_proj.weight.data.view(num_kv_heads, head_dim, -1)
            
            for kv_idx in range(num_kv_heads):
                group_q_heads = list(range(kv_idx * num_groups, (kv_idx + 1) * num_groups))
                if all(h in heads_to_prune for h in group_q_heads):
                    k_w[kv_idx].zero_()
                    v_w[kv_idx].zero_()
            
            # O_proj 处理 
            o_w = layer.self_attn.o_proj.weight.data.view(-1, num_heads, head_dim)
            for h in heads_to_prune:
                o_w[:, h, :].zero_()
                
            total_heads_pruned += len(heads_to_prune)
            
        # ------------------------------------------
        # 手术 B：切除 MLP 通道 (Channel-level)
        # ------------------------------------------
        mlp_status = ""
        if retention_dict[l_idx]['mlp_full_keep']:
            # 兼容老逻辑：要求整层满血保留
            mlp_status = "满血保留 (100%)"
        else:
            channels_to_keep = sorted(set(retention_dict[l_idx]['mlp_channels']))
            channels_to_prune = [c for c in range(intermediate_size) if c not in channels_to_keep]
            
            if len(channels_to_prune) == intermediate_size:
                # 整个 MLP 报废
                layer.mlp.gate_proj.weight.data.zero_()
                layer.mlp.up_proj.weight.data.zero_()
                layer.mlp.down_proj.weight.data.zero_()
                total_mlp_channels_pruned += intermediate_size
                mlp_status = "❌ 整层切除 (0%)"
                
            elif len(channels_to_prune) > 0:
                # ✂️ 精细切除部分通道
                # gate_proj 和 up_proj 切除行 (dim 0)
                for c in channels_to_prune:
                    layer.mlp.gate_proj.weight.data[c, :].zero_()
                    layer.mlp.up_proj.weight.data[c, :].zero_()
                    # down_proj 切除列 (dim 1)
                    layer.mlp.down_proj.weight.data[:, c].zero_()
                    
                total_mlp_channels_pruned += len(channels_to_prune)
                keep_ratio = len(channels_to_keep) / intermediate_size * 100
                mlp_status = f"🔪 精细剪枝 (保留 {keep_ratio:.1f}%)"
            else:
                mlp_status = "满血保留 (100%)"

        # 打印进度日志
        print(f"第 {l_idx:2d} 层: 砍掉 {len(heads_to_prune):2d}/{num_heads:2d} 个 Heads | MLP状态: {mlp_status}")

    print("="*70)
    print(f"✅ 手术汇总:")
    print(f"   - 共切除 {total_heads_pruned} 个 Attention Heads")
    print(f"   - 共切除 {total_mlp_channels_pruned} 个 MLP 神经元通道")
    print("="*70)

In [11]:
#N:M 结构化剪枝 的增强代码

In [12]:
def apply_nm_pruning(model, n=2, m=4):
    """
    极限加速版 N:M 结构化剪枝 (Vectorized)
    消除了所有内部的 Python for 循环，直接使用 GPU 矩阵运算。
    """
    print(f"\n🚀 开始执行极限加速版 {n}:{m} 结构化稀疏剪枝...")
    
    # 遍历所有层，加上进度条
    for l_idx, layer in tqdm(enumerate(model.model.layers), total=len(model.model.layers), desc="Pruning Layers"):
        
        gate_weight = layer.mlp.gate_proj.weight.data # [intermediate_size, hidden_size]
        up_weight = layer.mlp.up_proj.weight.data     # [intermediate_size, hidden_size]
        down_weight = layer.mlp.down_proj.weight.data # [hidden_size, intermediate_size]
        
        intermediate_size, hidden_size = gate_weight.shape
        
        # ==========================================
        # 1. 重塑形状 (将连续的 M 个通道作为一组)
        # 形状变为: [intermediate_size // m, m, hidden_size]
        # ==========================================
        gate_reshaped = gate_weight.view(intermediate_size // m, m, hidden_size)
        up_reshaped = up_weight.view(intermediate_size // m, m, hidden_size)
        
        # ==========================================
        # 2. 联合评估通道重要性 (Vectorized)
        # 分别计算 gate 和 up 每一行的 L2 范数，并相加作为该通道的联合重要性得分
        # 形状: [intermediate_size // m, m]
        # ==========================================
        gate_norms = gate_reshaped.norm(p=2, dim=2)
        up_norms = up_reshaped.norm(p=2, dim=2)
        combined_norms = gate_norms + up_norms 
        
        # ==========================================
        # 3. 找出 Top-N 并生成 Mask (Vectorized)
        # ==========================================
        # topk_idx 形状: [intermediate_size // m, n]
        _, topk_idx = combined_norms.topk(n, dim=1)
        
        # 创建全 0 掩码，形状: [intermediate_size // m, m]
        mask = torch.zeros_like(combined_norms, dtype=torch.bool)
        # 使用 scatter_ 在 topk 的位置填入 1 (True)
        mask.scatter_(1, topk_idx, True)
        
        # ==========================================
        # 4. 一键应用 Mask (就地修改内存，极致极速)
        # ==========================================
        # 扩展 mask 形状以匹配 gate 和 up: [intermediate_size // m, m, 1] -> [intermediate_size, 1]
        channel_mask = mask.view(intermediate_size, 1)
        
        # 直接使用掩码将不需要的通道归零
        gate_weight.mul_(channel_mask)
        up_weight.mul_(channel_mask)
        
        # 对于 down_proj，输入维度是 intermediate_size (对应矩阵的列)
        # channel_mask 需转置为 [1, intermediate_size] 以匹配列操作
        down_weight.mul_(channel_mask.t())

    print("\n✅ 所有网络层 N:M 剪枝完成！")

In [13]:
#用wiki数据集进行评估ppl   

In [14]:
def evaluate_ppl(model, tokenizer, seqlen=2048):
    
    # 2. 提取出 test 集
    testdata = dataset['test']
    
    # 论文标准拼接法：用双换行符连成一篇超长文
    text = "\n\n".join(testdata['text'])
    
    print("正在进行 Tokenization (这可能需要几秒钟)...")
    testenc = tokenizer(text, return_tensors='pt').input_ids
    
    nsamples = testenc.numel() // seqlen
    
    model.eval()
    nlls = []
    
    print(f"开始评估，共 {nsamples} 个文本块，块大小 {seqlen}...")
    with torch.no_grad():
        for i in tqdm(range(nsamples), desc="Evaluating PPL"):
            batch = testenc[:, i * seqlen : (i + 1) * seqlen].to(model.device)
            
            with torch.amp.autocast(device_type="cuda", dtype=torch.bfloat16):
                outputs = model(batch, labels=batch)
                loss = outputs.loss
            
            nlls.append(loss)
            
    avg_loss = torch.stack(nlls).mean()
    ppl = torch.exp(avg_loss).item()
    
    return ppl

In [15]:
#评估

In [16]:
def evaluate_acc(model, tokenizer, seqlen=2048):
    # 假设 testdata 和 text 拼接逻辑与你原代码一致
    testdata = dataset['test']
    text = "\n\n".join(testdata['text'])
    
    print("正在进行 Tokenization...")
    testenc = tokenizer(text, return_tensors='pt').input_ids
    
    # 确保序列长度能被 seqlen 整除
    nsamples = testenc.numel() // seqlen
    testenc = testenc[:, :nsamples * seqlen]
    
    model.eval()
    total_correct = 0
    total_tokens = 0
    
    print(f"开始评估准确率，共 {nsamples} 个文本块...")
    with torch.no_grad():
        for i in tqdm(range(nsamples), desc="Evaluating Accuracy"):
            batch = testenc[:, i * seqlen : (i + 1) * seqlen].to(model.device)
            
            # 使用 autocast 保持一致的精度
            with torch.amp.autocast(device_type="cuda", dtype=torch.bfloat16):
                outputs = model(batch)
                logits = outputs.logits # shape: [1, seqlen, vocab_size]
            
            # 获取模型预测概率最高的 token
            # shift_logits 和 shift_labels 对齐：
            # 模型在位置 t 预测的是 t+1，因此取 logits[:, :-1] 与 labels[:, 1:] 比较
            shift_logits = logits[:, :-1, :]
            shift_labels = batch[:, 1:]
            
            # 获取预测结果
            predictions = torch.argmax(shift_logits, dim=-1)
            
            # 计算准确数
            correct = (predictions == shift_labels).sum().item()
            total_correct += correct
            total_tokens += shift_labels.numel()
            
    acc = total_correct / total_tokens
    return acc

In [17]:
#将剪枝实验结果自动追加到 JSON 文件中

In [18]:
def append_result_to_json(json_path, retention_rate, ppl):
    # 确保保存的目录存在
    os.makedirs(os.path.dirname(json_path), exist_ok=True)
    
    # 将保留率转换为字符串作为 Key (例如 0.65 -> "0.65")
    key = str(retention_rate)
    
    # 准备要写入的数据字典，仅保留 ppl
    new_entry = {
        "ppl": round(ppl, 4)
    }
    
    # 1. 尝试读取已有的 JSON 文件
    data = {}
    if os.path.exists(json_path):
        with open(json_path, 'r', encoding='utf-8') as f:
            try:
                data = json.load(f)
            except json.JSONDecodeError:
                data = {}
    
    # 2. 更新或追加数据
    data[key] = new_entry
    
    # 3. 写回文件
    with open(json_path, 'w', encoding='utf-8') as f:
        json.dump(data, f, indent=4, ensure_ascii=False)
        
    print(f"实验结果已成功追加至: {json_path} [保留率: {key}]")

# --- Jupyter 运行示例 ---
# 假设你在 dataset_path 上完成了评估
# results_path = "./results/pruning_metrics.json"
# append_result_to_json(results_path, 0.65, 17.8896)

In [19]:
#Cell 6: 运行测试

In [ ]:
# 1. 加载数据
raw_data = torch.load(lcb_path)

# 2. 核心修复：将“字典列表”转换为“扁平字典”
# 假设你的数据结构是 [{'id': '...', 'lcb': ...}, ...]
if isinstance(raw_data, list) and len(raw_data) > 0 and isinstance(raw_data[0], dict):
    print("检测到复杂的字典列表结构，正在进行扁平化转换...")
    lcb_scores = {}
    for item in raw_data:
        # 自动适配不同的键名（id/unit_id 和 lcb/score）
        uid = item.get('id') or item.get('unit_id')
        score = item.get('lcb') or item.get('score') or item.get('value', 0)
        if uid:
            lcb_scores[uid] = score
else:
    lcb_scores = raw_data

print(f"✅ 数据解析成功！总单元数: {len(lcb_scores)}")

selected_ids = generate_config(
    lcb_scores_dict=lcb_scores, 
    target_sparsity=0.50,
    min_head_survival_rate=0.1,
    num_layers=model.config.num_hidden_layers,
    num_heads=model.config.num_attention_heads,
    intermediate_size=model.config.intermediate_size
)

# 记录实际保留率 (基于单元数量或参数量)
actual_retention_rate = len(selected_ids) / 1056 
print(f"贪心算法分配完成！实际提取单元: {len(selected_ids)}/1056 (保留率: {actual_retention_rate:.4f})")

# 4. 执行物理剪枝
# 移除低贡献且波动大的模型单元，优先保留 LCB 高的单元
print("正在对大模型执行结构化剪枝...")
# apply_pruning(model, selected_ids)
apply_nm_pruning(model,4,8)
# 5. 评估零样本 PPL
# 通过端到端验证，量化配置变更对性能的影响
print("开始执行评估 (PPL)...")
final_ppl = evaluate_ppl(model, tokenizer)

print("\n" + "="*50)
print(f"贪心鲁棒剪枝 PPL : {final_ppl:.4f}")
print("="*50 + "\n")

print("开始执行评估 (准确率acc)...")
final_acc = evaluate_acc(model, tokenizer)

print("\n" + "="*50)
print(f"准确率acc : {final_acc:.4f}")
print("="*50 + "\n")

# append_result_to_json(
#     json_path=json_path,
#     retention_rate=round(actual_retention_rate, 2), # 使用实际保留率作为 Key
#     ppl=final_ppl
# )

In [ ]:
#通过 plt.bar 展现各层的保留情况，可以清晰地证明你的“层级保护”和“频域均衡”策略是否按预期工作。

In [ ]:
def plot_layer_stats(selected_ids, total_layers=32):
    # 1. 数据统计
    heads_per_layer = np.zeros(total_layers)
    mlp_per_layer = np.zeros(total_layers)
    
    for uid in selected_ids:
        # 从 ID 中解析层号，处理各种格式 (如 layer_5_mlp 或 o_proj_head_12)
        parts = uid.replace('.', '_').split('_')
        layer_idx = -1
        for p in parts:
            if p.isdigit():
                layer_idx = int(p)
                break
        
        if layer_idx != -1:
            if "head" in uid:
                heads_per_layer[layer_idx] += 1
            elif "mlp" in uid:
                mlp_per_layer[layer_idx] += 1

    # 2. 绘图设置 (学术风格)
    plt.style.use('seaborn-v0_8-whitegrid') # 使用清晰的学术网格风格
    fig, ax = plt.subplots(figsize=(14, 6), dpi=100)
    
    layers = np.arange(total_layers)
    
    # 绘制堆叠柱状图
    p1 = ax.bar(layers, heads_per_layer, color='#3498db', label='Attention Heads (Max 32)', alpha=0.85, edgecolor='black', linewidth=0.5)
    p2 = ax.bar(layers, mlp_per_layer, bottom=heads_per_layer, color='#e74c3c', label='MLP Blocks (Max 1)', alpha=0.85, edgecolor='black', linewidth=0.5)

    # 3. 装饰图表
    ax.set_title(f"Layer-wise Retention Distribution (Total Retained: {len(selected_ids)})", fontsize=15, fontweight='bold')
    ax.set_xlabel("Layer Index", fontsize=12)
    ax.set_ylabel("Number of Retained Units", fontsize=12)
    ax.set_xticks(layers)
    ax.set_ylim(0, 36) # 留出一点空间放图例
    
    # 添加一条 100% 保留的参考线
    ax.axhline(y=33, color='gray', linestyle='--', alpha=0.5, label='Full Capacity (32H + 1M)')

    ax.legend(loc='upper right', frameon=True, facecolor='white')
    
    plt.tight_layout()
    
    plt.show()

# 运行绘图
plot_layer_stats(selected_ids)

In [ ]:
def plot_from_json(json_path):
    if not os.path.exists(json_path):
        print(f"❌ 错误：找不到文件 {json_path}")
        return

    # 1. 读取数据
    with open(json_path, 'r', encoding='utf-8') as f:
        raw_data = json.load(f)

    # 2. 解析新版 JSON 格式 (键即为保留率)
    # 提取所有实验点：x轴用保留率（float），y轴用PPL
    plot_data = []
    for rate_str, info in raw_data.items():
        plot_data.append({
            "rate": float(rate_str), 
            "ppl": info['ppl']
        })

    # 3. 按保留率从大到小排序 (确保从 1.0 到 0.5 的连线顺序正确)
    plot_data.sort(key=lambda x: x['rate'], reverse=True)

    rates = [d['rate'] for d in plot_data]
    ppls = [d['ppl'] for d in plot_data]

    # 4. 开始绘图
    plt.figure(figsize=(10, 6), dpi=120)
    
    # 绘制折线和点
    # 使用与文档研究背景一致的 Llama-2 标签 [cite: 9]
    plt.plot(rates, ppls, marker='o', linestyle='-', color='#1f77b4', 
             linewidth=2, markersize=8, label='Llama-2 Pruning Path (LCB-based)')

    # 在点上方标注 PPL 数值
    for i, val in enumerate(ppls):
        plt.annotate(f"{val:.2f}", (rates[i], ppls[i]), 
                     textcoords="offset points", xytext=(0, 10), 
                     ha='center', fontweight='bold', fontsize=9)
    
    # 在点下方标注保留率百分比 (例如 0.65 -> 65%)
    for i, rate in enumerate(rates):
        plt.annotate(f"{rate*100:.0f}%", (rates[i], ppls[i]), 
                     textcoords="offset points", xytext=(0, -20), 
                     ha='center', fontsize=8, color='gray')

    # 轴设置
    plt.gca().invert_xaxis()  # X轴从高保留率（1.0）降到低保留率 [cite: 40]
    plt.yscale('log')         # 使用对数坐标观察性能崩溃临界点 [cite: 42]
    plt.grid(True, which="both", ls="--", alpha=0.5)
    
    plt.xlabel('Retention Rate (Parameter Percentage)', fontsize=11)
    plt.ylabel('WikiText-2 PPL (Log Scale)', fontsize=11)
    plt.title('Pruning Sensitivity Analysis (LCB Robust Inference)', fontsize=13, pad=15)
    plt.legend()

    plt.tight_layout()
    plt.show()

# 执行绘图
plot_from_json(json_path)

In [ ]:
#生成Lcb权重文件

In [ ]:
# from scipy.fftpack import dct
# from sklearn.feature_selection import mutual_info_classif
# import warnings

# warnings.filterwarnings("ignore")

# class FrequencyMIAnalyzer:
#     def __init__(self, num_bands=3, num_loss_bins=10):
#         """
#         初始化频域互信息分析器
#         :param num_bands: B (频带数量)，例如 3 代表 低频、中频、高频
#         :param num_loss_bins: L (将连续损失离散化为多少个事件类别，对应公式 8-9)
#         """
#         self.num_bands = num_bands
#         self.num_loss_bins = num_loss_bins
        
#         # 用于在校准过程中收集数据的容器
#         # 结构: { unit_id: [ sample1_bands, sample2_bands, ... ] }
#         self.unit_band_energies = {} 
#         # 收集每条样本(序列)的代表性任务损失事件 Y'
#         self.task_events = []

#     def process_sequence(self, unit_id, grad_tensor, seq_losses):
#         """
#         处理单条序列(样本)的梯度和损失 (对应公式 1 到 7)
#         :param unit_id: 单元名称 (如 'layer_0_head_0')
#         :param grad_tensor: 该单元在该序列上的梯度矩阵 [Seq_len, Hidden_dim]
#         :param seq_losses: 该序列的 Token 级损失 [Seq_len]
#         """
#         if unit_id not in self.unit_band_energies:
#             self.unit_band_energies[unit_id] = []

#         # ---------------------------------------------------------
#         # 第一步：降维与标准化 (公式 2, 4)
#         # ---------------------------------------------------------
#         # 按特征维度求 L2 范数，得到一维的序列响应 g_u(t)
#         if isinstance(grad_tensor, torch.Tensor):
#             g_u = torch.norm(grad_tensor, p=2, dim=-1).cpu().numpy()
#         else:
#             g_u = np.linalg.norm(grad_tensor, axis=-1)

#         # 样本内标准化
#         g_u_norm = (g_u - np.mean(g_u)) / (np.std(g_u) + 1e-8)

#         # ---------------------------------------------------------
#         # 第二步：离散余弦变换 (DCT) 与能量谱计算 (公式 5, 6)
#         # ---------------------------------------------------------
#         # norm='ortho' 保证变换前后的能量守恒
#         dct_coeffs = dct(g_u_norm, type=2, norm='ortho')
#         energy_spectrum = np.abs(dct_coeffs) ** 2  # e_u(k)

#         # ---------------------------------------------------------
#         # 第三步：频带聚合 (公式 7, 13)
#         # ---------------------------------------------------------
#         # 论文中使用了极其复杂的贪心合并算法寻找边界，在实际复现中，
#         # 将频率均分为 num_bands 个粗粒度频带(低/中/高)即可达到 95% 的效果
#         seq_len = len(energy_spectrum)
#         band_size = max(1, seq_len // self.num_bands)
        
#         Z_bands = np.zeros(self.num_bands)
#         for b in range(self.num_bands):
#             start = b * band_size
#             # 最后一个频带包揽剩下的所有高频尾巴
#             end = seq_len if b == self.num_bands - 1 else (b + 1) * band_size
#             Z_bands[b] = np.sum(energy_spectrum[start:end])
            
#         self.unit_band_energies[unit_id].append(Z_bands)

#     def register_task_loss(self, seq_losses):
#         """
#         记录单条序列的任务损失 (用于后续提取任务事件 Y')
#         这里取平均损失作为该序列的总体任务表现
#         """
#         if isinstance(seq_losses, torch.Tensor):
#             mean_loss = torch.mean(seq_losses).item()
#         else:
#             mean_loss = np.mean(seq_losses)
#         self.task_events.append(mean_loss)

#     def finalize_mi_computation(self):
#         """
#         校准集跑完后，统一计算互信息！ (对应公式 8, 9, 14)
#         :return: 包含完整 mi_bands 的字典
#         """
#         print(f"🔄 开始在 {len(self.task_events)} 条校准数据上估计互信息 (MI)...")
        
#         # 1. 任务事件离散化 Y' (等距分箱法 binning)
#         loss_array = np.array(self.task_events)
#         bins = np.percentile(loss_array, np.linspace(0, 100, self.num_loss_bins + 1))
#         # 解决边界问题
#         bins[0] -= 1e-5 
#         bins[-1] += 1e-5
#         Y_prime = np.digitize(loss_array, bins)

#         final_results = []
        
#         # 2. 对每个单元，利用 sklearn 估计连续频带响应与离散任务事件之间的互信息
#         for unit_id, band_records in self.unit_band_energies.items():
#             # band_matrix: [Num_samples, Num_bands]
#             band_matrix = np.array(band_records) 
            
#             # 检查是否有异常值
#             if np.isnan(band_matrix).any():
#                 band_matrix = np.nan_to_num(band_matrix)

#             # 使用 kNN 互信息估计器 (对应论文 3.1 节，sklearn 底层正是基于近邻熵估计)
#             # discrete_features=False 表示输入特征 (频带能量) 是连续变量
#             mi_scores = mutual_info_classif(band_matrix, Y_prime, discrete_features=False)
            
#             # 计算总 LCB 分数 (论文公式 15：将各频带互信息加权求和，这里简化权重全为 1)
#             lcb_score = np.sum(mi_scores)
            
#             final_results.append({
#                 'id': unit_id,
#                 'lcb': float(lcb_score),
#                 'cost': 1.0,
#                 'mi_bands': mi_scores.tolist() # 这就是你梦寐以求的真实频段数据！
#             })
            
#         print("✅ 真实频带互信息计算完毕！")
#         return final_results

In [ ]:
# import torch.nn as nn
# from torch.utils.data import DataLoader
# from transformers import DataCollatorForLanguageModeling
# from tqdm import tqdm

# # ==========================================
# # 1. 定义全量梯度拦截器 (Hooker)
# # ==========================================
# class FullModelGradientHooker:
#     def __init__(self, model):
#         self.model = model
#         self.gradients = {}
#         self.hooks = []

#     def _get_hook(self, name):
#         def hook(module, grad_input, grad_output):
#             # grad_output[0] 形状: [batch, seq, hidden]
#             self.gradients[name] = grad_output[0].detach().cpu()
#         return hook

#     def register(self):
#         # 清除旧钩子防止冲突
#         self.remove()
#         for i, layer in enumerate(self.model.model.layers):
#             # 拦截 Attention Head 信号 (通过 o_proj)
#             self.hooks.append(
#                 layer.self_attn.o_proj.register_full_backward_hook(self._get_hook(f"layer_{i}_heads"))
#             )
#             # 拦截 MLP 信号 (通过 down_proj)
#             self.hooks.append(
#                 layer.mlp.down_proj.register_full_backward_hook(self._get_hook(f"layer_{i}_mlp"))
#             )
#         print(f"✅ 已成功注册 {len(self.hooks)} 个拦截点 (1024 Heads + 32 MLPs)")

#     def remove(self):
#         for h in self.hooks:
#             h.remove()
#         self.hooks = []
#         self.gradients.clear()

# # ==========================================
# # 2. 核心：严谨校准主函数
# # ==========================================
# def run_rigorous_calibration(model, dataloader, analyzer):
#     hooker = FullModelGradientHooker(model)
#     hooker.register()
    
#     # 开启训练模式以允许梯度回传，并确保梯度开启
#     model.train() 
#     for param in model.parameters():
#         param.requires_grad = True
        
#     num_heads = model.config.num_attention_heads
#     head_dim = model.config.hidden_size // num_heads

#     print("⚡ 开始执行 1056 单元深度频域互信息分析...")
    
#     try:
#         for step, batch in enumerate(tqdm(dataloader, desc="Calibration Steps")):
#             # 将数据移动到设备
#             input_ids = batch['input_ids'].to(model.device)
#             attention_mask = batch['attention_mask'].to(model.device)
#             labels = input_ids.clone()
            
#             model.zero_grad()

#             # --- 前向传播 ---
#             outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
#             loss = outputs.loss
            
#             # --- 计算 Token 级 Loss (用于 MI 计算) ---
#             with torch.no_grad():
#                 # 错位对齐：预测下一个 token，长度变为 seq_len - 1
#                 logits = outputs.logits[..., :-1, :].contiguous()
#                 shift_labels = labels[..., 1:].contiguous()
                
#                 token_losses = torch.nn.functional.cross_entropy(
#                     logits.view(-1, logits.size(-1)), 
#                     shift_labels.view(-1), 
#                     reduction='none'
#                 ).view(1, -1)
                
#                 loss_np = token_losses.cpu().float().numpy()
#                 analyzer.register_task_loss(loss_np)

#             # --- 反向传播 (触发 Hooks) ---
#             loss.backward()

#             # --- 处理 1056 个单元的梯度信号 ---
#             for i in range(len(model.model.layers)):
#                 # A. 处理 32 个 Heads
#                 h_grad = hooker.gradients[f"layer_{i}_heads"] 
#                 # 关键：梯度长度也要对齐 loss (:-1)
#                 h_grad = h_grad[:, :-1, :].view(1, -1, num_heads, head_dim)
                
#                 for h_idx in range(num_heads):
#                     u_id = f"layer_{i}_head_{h_idx}"
#                     g_seq = h_grad[0, :, h_idx, :].float().numpy()
#                     analyzer.process_sequence(u_id, g_seq, loss_np)

#                 # B. 处理 1 个 MLP
#                 m_grad = hooker.gradients[f"layer_{i}_mlp"]
#                 m_grad = m_grad[:, :-1, :].float().numpy()
#                 u_id = f"layer_{i}_mlp"
#                 analyzer.process_sequence(u_id, m_grad[0], loss_np)
                
#     finally:
#         hooker.remove() # 报错也要移除钩子防止内存泄漏
        
#     print("🎨 执行最终互信息聚合...")
#     return analyzer.finalize_mi_computation()

# # ==========================================
# # 3. 准备运行环境并启动
# # ==========================================

# # A. 处理数据集 (解决 DatasetDict 和 Tokenization 问题)
# def prepare_dataloader(dataset, tokenizer, num_samples=128, max_length=512):
#     # 自动识别 DatasetDict 结构
#     target_ds = dataset['train'] if isinstance(dataset, dict) or 'train' in dataset.keys() else dataset
    
#     # 1. 选取子集
#     subset = target_ds.select(range(min(num_samples, len(target_ds))))
    
#     # 2. Tokenize 处理
#     def tokenize_fn(examples):
#         return tokenizer(examples["text"], padding="max_length", truncation=True, max_length=max_length)
    
#     print("⏳ 正在对校准集进行 Tokenization...")
#     tokenized_ds = subset.map(tokenize_fn, batched=True, remove_columns=target_ds.column_names)
    
#     # 3. 创建 DataLoader
#     collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)
#     return DataLoader(tokenized_ds, batch_size=1, shuffle=False, collate_fn=collator)

# # B. 初始化
# calib_loader = prepare_dataloader(dataset, tokenizer)
# # 假设 FrequencyMIAnalyzer 类已经在之前的 Cell 中定义好
# # analyzer = FrequencyMIAnalyzer(num_bands=3, num_loss_bins=10)

# # C. 执行严谨计算
# print("🚀 启动 1056 单元全量分析流程...")
# final_real_data = run_rigorous_calibration(
#     model=model, 
#     dataloader=calib_loader, 
#     analyzer=analyzer
# )

# output_file = "./results/lcb_scores_v4.pt"
# torch.save(final_real_data, output_file)

# print(f"🎉 任务圆满完成！")
# print(f"📊 1056 单元严谨评分文件已保存至: {output_file}")